# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "The Anatomy of Growing Content"

**Claim:** Pages trending upward (74,187) are 37.6% longer and 20% younger than pages trending downward (45,272); the paper reports this as a large-sample, directionally robust comparison.

**Where the label comes from:** "up" vs "down" is a 30-day trend-direction bucket, the same kind of proxy label our own model uses. The paper is upfront that this is observational, not causal.

**My methodology question:** the paper's own Limitations section separately states "content age confounds model-performance comparisons," but Finding 1 reports word count and age as two independent differences between growing and declining content without addressing whether they're actually confounded with each other, older pages might simply have been written in an earlier, shorter-content era of the site, rather than age and length being two separate causal levers. Does the analysis (or a follow-up) hold age constant while comparing word count, or vice versa, to check whether one of these two "differences" is actually just a proxy for the other? This doesn't undercut the finding, it's exactly the kind of check that would make an already-strong sample-size result even more defensible.

### Finding 2: "The Freshness Multiplier"

**Claim:** 365+ day content refreshed within 30 days shows a 3.2x health-score boost and 57x more impressions, framed as "one of the strongest measured levers available."

**Where the label comes from:** health score is the paper's own composite metric (impressions + position + CTR + scroll depth), and "refreshed" is presumably a binary flag on whether an edit happened in the last 30 days.

**My methodology question:** this compares refreshed vs. non-refreshed pages within the 365+ bucket, but doesn't say whether refresh timing was randomly assigned or editor-selected. If editors tend to refresh pages they already judge to have recovery potential (existing backlinks, a topic still in demand), the 57x impression gap could partly reflect that selection, not the refresh action itself. The paper is careful elsewhere (it explicitly flags the 361+ bucket's 283:1 ratio as unstable, "only 1 declining page"), so this isn't a paper that hides its caveats, this specific comparison just doesn't state whether refresh assignment was closer to random or closer to editor-chosen, and that distinction changes how strongly "refresh timing is one of the strongest measured levers" should be read.

**Constructive note, tying back to our own work:** both questions above are versions of a check we had to run on our own model this week, whether an apparent driver (age, refresh timing) is really doing the work, or is a stand-in for something else nearby in time. Our own `content_age_days` feature turned out to be a disguised calendar-month signal rather than genuine per-page aging, worth naming here since it's the same category of question, not a "gotcha" specific to this paper.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
!pip install -q duckdb huggingface_hub lightgbm

In [3]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    import getpass
    login(token=getpass.getpass("Paste HF token: "))

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Download dim_content and all 18 monthly fact files, sequentially with retry to avoid HF rate limits
import time
from huggingface_hub import hf_hub_download

dim_content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                     filename="dim_content.parquet")

all_months = [f"2025-{m:02d}" for m in range(1, 13)] + [f"2026-{m:02d}" for m in range(1, 7)]
all_files = []

for m in all_months:
    for attempt in range(3):
        try:
            path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet")
            all_files.append(path)
            print(f"Downloaded {m}")
            time.sleep(2)
            break
        except Exception as e:
            print(f"Retry {attempt+1} for {m}: {e}")
            time.sleep(10)

print(f"\nTotal files downloaded: {len(all_files)} of {len(all_months)}")


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded 2025-01


...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded 2026-05


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded 2026-06

Total files downloaded: 18 of 18


In [7]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

query = f"""
WITH content_meta AS (
    SELECT
        content_hash_id, content_type, main_intent, competition_level, provider_used,
        search_volume, competition, cpc, backlinks, category_count, word_count, char_count,
        content_created_date
    FROM read_parquet('{dim_content_path}')
    WHERE is_published = TRUE AND is_deleted = FALSE
),
base AS (
    SELECT
        client_hash_id, content_hash_id, report_date, ga4_data_available,
        gsc_impressions, gsc_clicks, gsc_sum_position,
        ga4_engaged_sessions, ga4_total_engagement_sec,
        sessions_organic, scroll_events
    FROM read_parquet({all_files})
    WHERE gsc_data_available = TRUE
      AND client_hash_id != 'client_08a6a72ff48e62c0'
),
rolled AS (
    SELECT
        *,
        SUM(gsc_impressions) OVER w90 AS impressions_90d,
        SUM(gsc_clicks) OVER w90 AS clicks_90d,
        SUM(gsc_sum_position) OVER w90 AS sum_position_90d,
        SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions ELSE NULL END) OVER w90 AS engaged_sessions_90d,
        SUM(CASE WHEN ga4_data_available THEN ga4_total_engagement_sec ELSE NULL END) OVER w90 AS engagement_sec_90d,
        SUM(CASE WHEN ga4_data_available THEN sessions_organic ELSE NULL END) OVER w90 AS sessions_organic_90d,
        SUM(CASE WHEN ga4_data_available THEN scroll_events ELSE NULL END) OVER w90 AS scroll_events_90d,
        STDDEV(gsc_sum_position / NULLIF(gsc_impressions, 0)) OVER w7 AS position_roll7_std,
        SUM(gsc_impressions) OVER w30 AS impressions_last30_feature,
        SUM(gsc_impressions) OVER wfut AS impressions_next30,
        ROW_NUMBER() OVER (
            PARTITION BY content_hash_id, DATE_TRUNC('month', report_date)
            ORDER BY report_date DESC
        ) AS rn_in_month
    FROM base
    WINDOW
        w90  AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 89 DAYS PRECEDING AND CURRENT ROW),
        w30  AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 29 DAYS PRECEDING AND CURRENT ROW),
        w7   AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW),
        wfut AS (PARTITION BY content_hash_id ORDER BY report_date RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING)
),
snapshots AS (
    SELECT * FROM rolled WHERE rn_in_month = 1
)
SELECT
    s.*,
    m.content_type, m.main_intent, m.competition_level, m.provider_used,
    m.search_volume, m.competition, m.cpc, m.backlinks, m.category_count,
    m.word_count, m.char_count,
    DATE_DIFF('day', m.content_created_date, s.report_date) AS content_age_days
FROM snapshots s
JOIN content_meta m USING (content_hash_id)
WHERE DATE_DIFF('day', m.content_created_date, s.report_date) >= 90
  AND s.impressions_90d > 0
  AND s.impressions_next30 IS NOT NULL
"""

df = con.execute(query).df()
print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(663190, 34)


In [8]:
df["report_date"] = pd.to_datetime(df["report_date"])
df = df[(df["report_date"] >= pd.Timestamp("2025-05-01")) & (df["report_date"] <= pd.Timestamp("2026-04-30"))].copy()
print(df.shape)

(552510, 34)


In [9]:
eligible = df["impressions_last30_feature"] >= 100

trend_pct = np.where(
    df["impressions_last30_feature"] > 0,
    (df["impressions_next30"] - df["impressions_last30_feature"]) / df["impressions_last30_feature"] * 100,
    np.nan
)
df["trend_pct"] = trend_pct
df["trend_direction"] = np.select(
    [~eligible, df["trend_pct"] > 20, df["trend_pct"] < -20],
    ["insufficient_volume", "up", "down"],
    default="stable"
)
df["is_declining_label"] = (df["trend_direction"] == "down").astype("int8")

model_df = df[df["trend_direction"] != "insufficient_volume"].copy()
print(f"Eligible rows: {len(model_df)} of {len(df)}")

Eligible rows: 357671 of 552510


In [10]:
model_df["ctr_90d"] = model_df["clicks_90d"] / model_df["impressions_90d"] * 100
model_df["avg_position_90d"] = model_df["sum_position_90d"] / model_df["impressions_90d"]

cat_cols = ["content_type", "main_intent", "competition_level", "provider_used"]
for c in cat_cols:
    model_df[c] = model_df[c].fillna("unknown").astype("category")

model_df_encoded = pd.get_dummies(model_df, columns=cat_cols, drop_first=True)

numeric_features = [
    "impressions_90d", "clicks_90d", "ctr_90d", "avg_position_90d",
    "engaged_sessions_90d", "engagement_sec_90d", "sessions_organic_90d", "scroll_events_90d",
    "position_roll7_std", "search_volume", "competition", "cpc",
    "backlinks", "category_count", "word_count", "char_count"
]
dummy_cols = [c for c in model_df_encoded.columns if any(c.startswith(f"{cc}_") for cc in cat_cols)]
features = numeric_features + dummy_cols

print(f"Total features: {len(features)}")
print(f"Rows: {len(model_df_encoded)}")

Total features: 28
Rows: 357671


In [11]:
model_df_encoded.to_parquet('/content/drive/MyDrive/flyrank_snapshots.parquet')

import json
with open('/content/drive/MyDrive/flyrank_meta.json', 'w') as f:
    json.dump({"features": features}, f)

print("Saved.")

Saved.


In [14]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

# BEFORE: naive split (content-level grouping, which we later found still leaks via same-client similarity)
gss_before = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss_before.split(model_df_encoded, groups=model_df_encoded["content_hash_id"]))
tr_b, te_b = model_df_encoded.iloc[tr_idx], model_df_encoded.iloc[te_idx]

gbm_before = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31,
                                  learning_rate=0.05, is_unbalance=True, random_state=42, verbosity=-1)
gbm_before.fit(tr_b[features], tr_b["is_declining_label"])
te_b = te_b.copy()
te_b["risk"] = gbm_before.predict_proba(te_b[features])[:, 1]
p50_before = te_b.nlargest(50, "risk")["is_declining_label"].mean()

# AFTER: client-level holdout, averaged across 5 seeds (the honest split)
print(f"BEFORE (content-grouped split): Precision@50 = {p50_before:.3f}")

seeds = [42, 7, 123, 2024, 99]
scores = []
spike_months = ["2026-03", "2026-04"]

for seed in seeds:
    gss_s = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(gss_s.split(model_df_encoded, groups=model_df_encoded["client_hash_id"]))
    tr, te = model_df_encoded.iloc[tr_idx], model_df_encoded.iloc[te_idx]

    te = te[~te["report_date"].dt.to_period("M").astype(str).isin(spike_months)]
    if len(te) < 50:
        continue

    gbm_s = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31,
                                 learning_rate=0.05, is_unbalance=True, random_state=42, verbosity=-1)
    gbm_s.fit(tr[features], tr["is_declining_label"])
    te = te.copy()
    te["risk"] = gbm_s.predict_proba(te[features])[:, 1]
    p50 = te.nlargest(50, "risk")["is_declining_label"].mean()
    scores.append(p50)

print(f"AFTER, per seed: {[f'{s:.3f}' for s in scores]}")
print(f"AFTER (client-holdout, mean of 5 seeds): Precision@50 = {np.mean(scores):.3f} (σ={np.std(scores):.3f})")

BEFORE (content-grouped split): Precision@50 = 0.940
AFTER, per seed: ['0.880', '0.820', '0.560', '0.900', '0.780']
AFTER (client-holdout, mean of 5 seeds): Precision@50 = 0.788 (σ=0.122)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
gss_leak = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss_leak.split(model_df_encoded, groups=model_df_encoded["client_hash_id"]))

# Deliberate leak, same demonstration as w03, run on the FINAL feature set this time
leak_test = model_df_encoded.copy()
leak_test["leaky_feature"] = leak_test["is_declining_label"]

leaky_features = features + ["leaky_feature"]
tr_l, te_l = leak_test.iloc[train_idx], leak_test.iloc[test_idx]

rf_leaky = RandomForestClassifier(n_estimators=50, random_state=42).fit(tr_l[leaky_features].fillna(0), tr_l["is_declining_label"])
rf_honest = RandomForestClassifier(n_estimators=50, random_state=42).fit(tr_l[features].fillna(0), tr_l["is_declining_label"])

print(f"Accuracy WITH leaked feature: {rf_leaky.score(te_l[leaky_features].fillna(0), te_l['is_declining_label']):.3f}")
print(f"Accuracy WITHOUT (honest): {rf_honest.score(te_l[features].fillna(0), te_l['is_declining_label']):.3f}")

Accuracy WITH leaked feature: 1.000
Accuracy WITHOUT (honest): 0.566


Accuracy jumps to a suspicious 1.000 with the injected label-derived feature, and drops to a believable 0.566 without it, confirming the leak mechanic. (Note: raw accuracy is used here only to demonstrate leakage; Precision@50, the project's actual success metric, is a stronger indicator of real model quality on this imbalanced, client-shifted dataset.)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (too bold): "Our model predicts which pages will lose search traffic."

Rewritten (safe): "Under a client-holdout validation design, the model's top-50 ranked pages showed an average Precision@50 of 78.8% (σ=0.124 across 5 random splits) for identifying pages with observed declining search impressions in the 30 days following the feature window. This is a decision-support ranking, not a guarantee, individual predictions should be reviewed by a human before action is taken, and performance during anomalous periods (e.g. the March-April 2026 spike, excluded from this evaluation) is not yet established."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.